# Week 5 participant practical: choosing a filtered construction for the observed data type

This is the student investigation, built around a decision clinic comparing a state cloud, weighted interactions and a scalar field. The setup and mathematical framing are supplied. You must record predictions, complete short computational steps, check intermediate objects and justify an interpretation.

This laboratory uses one software stack, GUDHI, for three input types: a point cloud, a weighted graph and a gridded scalar field. The purpose is not API coverage. It is to keep asking which observed object, complex and filtration answer the question.

All homology uses $\mathbb F_2$. See the **Applied glossary** for *simplex tree*, *cubical complex*, *lower-star filtration* and *clique complex*.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Optional extensions come only after the core checkpoints agree.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
import gudhi as gd

def finite_intervals(st,dim):
    D=st.persistence_intervals_in_dimension(dim)
    return D[np.isfinite(D[:,1])] if len(D) else D

def longest(D): return float(np.max(D[:,1]-D[:,0])) if len(D) else 0.0
print('GUDHI',gd.__version__)

## 1. Observe: participant checkpoint

Three data objects are shown below: sampled states in a point cloud, weighted pairwise interactions in a graph, and a scalar field on a grid. Before computing, name what is directly observed and what must be constructed.

In [ ]:
theta=np.linspace(0,2*np.pi,70,endpoint=False)
cloud=np.c_[np.cos(theta),np.sin(theta)]+.055*RNG.normal(size=(70,2))

graph_edges={(0,1):.3,(1,2):.4,(0,2):.45,(2,3):.7,(3,4):.8,(2,4):.85}

x=np.linspace(-2,2,45); X,Y=np.meshgrid(x,x)
field=(X**2+Y**2-1.0)**2

fig,axes=plt.subplots(1,2,figsize=(8,3.3))
axes[0].scatter(*cloud.T,s=12); axes[0].set_aspect('equal'); axes[0].set_title('sampled state cloud')
axes[1].imshow(field,origin='lower',extent=[-2,2,-2,2]); axes[1].set_title('scalar field')
plt.show()

## 2. Predict: participant checkpoint

1. Which $H_1$ feature should the point cloud contain?
2. In the graph, should a three-clique remain a graph cycle or be filled as a 2-simplex?
3. For the field, what do low sublevel values represent geometrically?
4. Which maximum homology dimension and filtration cutoff are actually needed?

## 3. Implement: participant checkpoint

### A. Point cloud: Rips filtration

GUDHI uses the pairwise-distance threshold as the Rips filtration value. Construct only through dimension 2 because that is enough to calculate $H_1$.

In [ ]:
rips=gd.RipsComplex(points=cloud,max_edge_length=1.8)
st_point=rips.create_simplex_tree(max_dimension=2)
st_point.compute_persistence(homology_coeff_field=2)
D1_point=finite_intervals(st_point,1)
print('simplices:',st_point.num_simplices(),'longest finite H1:',round(longest(D1_point),3))

### B. Weighted graph: graph or clique complex?

Keep vertices and edge weights fixed. First retain only the graph, then expand every clique through dimension 2.

**Checkpoint.** Predict which version can fill the cycle on vertices $0,1,2$. After running, check that both versions contain the same six edges before comparing $H_1$.

In [ ]:
def weighted_graph_tree(fill_cliques):
    st = gd.SimplexTree()
    for vertex in range(5):
        st.insert([vertex], filtration=0.0)
    for edge, weight in graph_edges.items():
        st.insert(edge, filtration=weight)
    if fill_cliques:
        st.expansion(2)
    st.make_filtration_non_decreasing()
    st.compute_persistence(homology_coeff_field=2, persistence_dim_max=True)
    return st

# TODO: change only this switch, then compare the two stored results.
versions = {'graph only': False, 'clique complex': True}
graph_results = {}
for name, fill_cliques in versions.items():
    tree = weighted_graph_tree(fill_cliques)
    graph_results[name] = tree
    print(name, 'simplices:', tree.num_simplices(),
          'H1:', tree.persistence_intervals_in_dimension(1))

### C. Scalar field: cubical sublevel filtration

The top-dimensional cells carry field values and low values enter first.

**Checkpoint.** Inspect the image and predict whether the low-valued annulus should create a finite $H_1$ interval. Then complete the two method calls below. Negating the field is reserved for the comparison section because it changes the question.

In [ ]:
cubical = gd.CubicalComplex(top_dimensional_cells=field)
# TODO: call compute_persistence with coefficient field 2.
# TODO: assign the H0 and H1 interval arrays.
# Hint: cubical.persistence_intervals_in_dimension(dimension)
H0_cubical = None
H1_cubical = None
print('cubical dimension:', cubical.dimension())

## 4. Compare: participant checkpoint

Choose **one** controlled comparison first. Do not change all three inputs at once.

| Route | Keep fixed | Change | Type of question |
|---|---|---|---|
| Point cloud | coordinates and metric | sample density | numerical sensitivity |
| Weighted graph | vertices, edges and weights | clique filling | representation assumption |
| Scalar field | grid and values | sublevel versus superlevel | scientific question |

After completing one route, attempt a second as an extension.

In [ ]:
comparison_route = 'point cloud'  # choose: point cloud, graph, or field

if comparison_route == 'point cloud':
    small = cloud[::2]
    # TODO: construct and compute the subsampled Rips persistence.
    print('full sample size:', len(cloud), 'subsample size:', len(small))
elif comparison_route == 'graph':
    for name, tree in graph_results.items():
        print(name, tree.persistence_intervals_in_dimension(1))
elif comparison_route == 'field':
    # TODO: construct cubical persistence for -field and compare its meaning.
    print('field ranges:', (field.min(), field.max()), (-field).min(), (-field).max())

## 5. Interpret: participant checkpoint

1. Which output belongs to a point-cloud representation, which to an interaction model, and which to a field?
2. When is clique filling scientifically defensible?
3. What is the physical meaning of a cubical birth value?
4. Which resource limits should be reported?
5. What simpler baseline belongs beside each topology calculation?

**† Qualification.** Every example is synthetic. The workflows demonstrate consequences of choices, not empirical performance.